# Train Value Transformer In Colab

This notebook:

1. clones the GitHub repository,
2. mounts Google Drive,
3. unpacks `value_lichess_1m.zip` into `/content`,
4. trains the scalar-value ViT model,
5. saves the checkpoint back to Drive.

## GPU recommendation

For this project the model is small, about 6.34M parameters, and the processed dataset is about 307 MB. That means:

- `L4` is the best default choice: fast enough, modern, and not wasteful.
- `A100` is the best choice if you only care about finishing training as fast as possible.
- `T4` will work, but training will be noticeably slower.
- `H100` is overkill for this model unless it is free for you.

Practical recommendation: use `L4`; use `A100` if available at the same cost.

In [ ]:
# Repository and data paths
REPO_URL = "https://github.com/SadreevAmir/Transfromer_chess.git"
REPO_DIR = "/content/Transfromer_chess"
DATASET_ZIP_DRIVE_PATH = "/content/drive/MyDrive/value_lichess_1m.zip"
DATASET_DIR = "/content/data/processed/value_lichess_1m"
ARTIFACT_DIR = "/content/artifacts"
DRIVE_ARTIFACT_DIR = "/content/drive/MyDrive/transformer_chess_artifacts"

# Training configuration
TRAINING_CONFIG = {
    "epochs": 3,
    "batch_size": 256,
    "num_workers": 4,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "seed": 0,
    "max_train_batches": None,
    "max_val_batches": None,
    "output_name": "value_model_colab.pt",
    "use_amp": True,
}

# Model configuration
MODEL_CONFIG = {
    "d_model": 256,
    "depth": 8,
    "num_heads": 8,
    "mlp_ratio": 4,
    "dropout": 0.1,
}

print(TRAINING_CONFIG)
print(MODEL_CONFIG)


In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys
import importlib.util

repo_dir = Path(REPO_DIR)
src_dir = repo_dir / "src"
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=True)
os.chdir(REPO_DIR)
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

spec = importlib.util.find_spec("transformer_chess")
assert spec is not None, "transformer_chess is still not importable after clone/install"
print("repo ready:", repo_dir)
print("src dir:", src_dir)


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import shutil
from pathlib import Path

dataset_zip = Path(DATASET_ZIP_DRIVE_PATH)
dataset_dir = Path(DATASET_DIR)
dataset_root = dataset_dir.parent.parent
dataset_root.mkdir(parents=True, exist_ok=True)

if dataset_dir.exists():
    shutil.rmtree(dataset_dir)

shutil.unpack_archive(str(dataset_zip), str(dataset_root))
print("dataset unpacked to:", dataset_dir)


In [ ]:
from __future__ import annotations

import json
import math
import platform
import random
import subprocess
import time
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from torch import nn
from tqdm.auto import tqdm

from transformer_chess import (
    BoardValueTransformer,
    ValueTransformerConfig,
    load_manifest,
    make_dataloader,
    make_optimizer,
)


def detect_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def gpu_name() -> str:
    if not torch.cuda.is_available():
        return "CPU"
    try:
        return subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total",
                "--format=csv,noheader",
            ],
            text=True,
        ).strip()
    except Exception:
        return torch.cuda.get_device_name(0)


def print_runtime_diagnostics() -> None:
    print("platform:", platform.platform())
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    print("gpu:", gpu_name())


def summarize_manifest(dataset_dir: Path) -> None:
    manifest = load_manifest(dataset_dir)
    print("dataset_dir:", dataset_dir)
    print("source:", manifest.source)
    print("samples:", manifest.num_samples)
    print("train:", manifest.num_train)
    print("val:", manifest.num_val)
    print("clip_pawns:", manifest.clip_pawns)
    print("top_k:", manifest.top_k)
    print("min_pvs:", manifest.min_pvs)
    print("num_shards:", len(manifest.shards))


def render_history(history: list[dict]) -> None:
    if not history:
        return

    epochs = [row["epoch"] for row in history]
    train_loss = [row["train_loss"] for row in history]
    val_loss = [row["val_loss"] for row in history]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=epochs, y=train_loss, mode="lines+markers", name="train_loss"))
    fig.add_trace(go.Scatter(x=epochs, y=val_loss, mode="lines+markers", name="val_loss"))
    fig.update_layout(
        title="Training History",
        xaxis_title="Epoch",
        yaxis_title="MSE Loss",
        template="plotly_white",
        hovermode="x unified",
    )
    display(fig)


def run_train_epoch_with_progress(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    loader,
    *,
    device: torch.device,
    scaler: torch.amp.GradScaler | None,
    use_amp: bool,
    max_batches: int | None = None,
    desc: str = "train",
) -> tuple[float, int, int]:
    criterion = nn.MSELoss()
    model.train()
    total_loss = 0.0
    total_batches = 0
    total_samples = 0
    progress = tqdm(loader, total=max_batches, desc=desc, leave=False)
    amp_enabled = use_amp and device.type == "cuda"
    for batch_index, (inputs, targets) in enumerate(progress, start=1):
        batch_samples = int(inputs.shape[0])
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            predictions = model(inputs)
            loss = criterion(predictions, targets)

        if amp_enabled and scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        loss_value = float(loss.detach().item())
        total_loss += loss_value
        total_batches += 1
        total_samples += batch_samples
        progress.set_postfix(batch=batch_index, loss=f"{loss_value:.4f}", avg=f"{total_loss / total_batches:.4f}")
        if max_batches is not None and batch_index >= max_batches:
            break
    progress.close()
    return total_loss / total_batches, total_batches, total_samples


@torch.no_grad()
def run_eval_with_progress(
    model: nn.Module,
    loader,
    *,
    device: torch.device,
    use_amp: bool,
    max_batches: int | None = None,
    desc: str = "val",
) -> tuple[float, int, int]:
    criterion = nn.MSELoss()
    model.eval()
    total_loss = 0.0
    total_batches = 0
    total_samples = 0
    progress = tqdm(loader, total=max_batches, desc=desc, leave=False)
    amp_enabled = use_amp and device.type == "cuda"
    for batch_index, (inputs, targets) in enumerate(progress, start=1):
        batch_samples = int(inputs.shape[0])
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            predictions = model(inputs)
            loss_value = float(criterion(predictions, targets).item())
        total_loss += loss_value
        total_batches += 1
        total_samples += batch_samples
        progress.set_postfix(batch=batch_index, loss=f"{loss_value:.4f}", avg=f"{total_loss / total_batches:.4f}")
        if max_batches is not None and batch_index >= max_batches:
            break
    progress.close()
    return total_loss / total_batches, total_batches, total_samples


if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print_runtime_diagnostics()


In [ ]:
set_seed(TRAINING_CONFIG["seed"])
device = detect_device()
dataset_dir = Path(DATASET_DIR)
artifact_dir = Path(ARTIFACT_DIR)
artifact_dir.mkdir(parents=True, exist_ok=True)

manifest = load_manifest(dataset_dir)
summarize_manifest(dataset_dir)

train_steps = math.ceil(manifest.num_train / TRAINING_CONFIG["batch_size"])
val_steps = math.ceil(manifest.num_val / TRAINING_CONFIG["batch_size"]) if manifest.num_val else 0
effective_train_steps = min(train_steps, TRAINING_CONFIG["max_train_batches"]) if TRAINING_CONFIG["max_train_batches"] is not None else train_steps
effective_val_steps = min(val_steps, TRAINING_CONFIG["max_val_batches"]) if TRAINING_CONFIG["max_val_batches"] is not None else val_steps

train_loader = make_dataloader(
    dataset_dir,
    split="train",
    batch_size=TRAINING_CONFIG["batch_size"],
    num_workers=TRAINING_CONFIG["num_workers"],
    seed=TRAINING_CONFIG["seed"],
    shuffle=True,
)

val_loader = make_dataloader(
    dataset_dir,
    split="val",
    batch_size=TRAINING_CONFIG["batch_size"],
    num_workers=TRAINING_CONFIG["num_workers"],
    seed=TRAINING_CONFIG["seed"],
    shuffle=False,
)

config = ValueTransformerConfig(**MODEL_CONFIG)
model = BoardValueTransformer(config).to(device)
optimizer = make_optimizer(
    model,
    lr=TRAINING_CONFIG["lr"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
)
scaler = torch.amp.GradScaler("cuda") if TRAINING_CONFIG["use_amp"] and device.type == "cuda" else None

print("device:", device)
print("gpu:", gpu_name())
print("model_parameters:", sum(p.numel() for p in model.parameters()))
print("train_steps_per_epoch:", effective_train_steps)
print("val_steps_per_epoch:", effective_val_steps)


In [ ]:
history = []
output_path = artifact_dir / TRAINING_CONFIG["output_name"]

for epoch in range(TRAINING_CONFIG["epochs"]):
    print(f"\nEpoch {epoch + 1}/{TRAINING_CONFIG['epochs']}")
    started = time.perf_counter()

    train_loss, train_batches_done, train_samples_done = run_train_epoch_with_progress(
        model,
        optimizer,
        train_loader,
        device=device,
        scaler=scaler,
        use_amp=TRAINING_CONFIG["use_amp"],
        max_batches=TRAINING_CONFIG["max_train_batches"],
        desc=f"train {epoch + 1}/{TRAINING_CONFIG['epochs']}",
    )

    val_loss, val_batches_done, val_samples_done = run_eval_with_progress(
        model,
        val_loader,
        device=device,
        use_amp=TRAINING_CONFIG["use_amp"],
        max_batches=TRAINING_CONFIG["max_val_batches"],
        desc=f"val {epoch + 1}/{TRAINING_CONFIG['epochs']}",
    )

    elapsed = time.perf_counter() - started
    row = {
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "elapsed_seconds": float(elapsed),
        "train_batches": train_batches_done,
        "train_samples": train_samples_done,
        "val_batches": val_batches_done,
        "val_samples": val_samples_done,
    }
    history.append(row)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "model_config": MODEL_CONFIG,
            "training_config": TRAINING_CONFIG,
            "device": str(device),
            "history": history,
            "dataset_manifest": manifest,
            "clip_pawns": manifest.clip_pawns,
        },
        output_path,
    )

    clear_output(wait=True)
    print_runtime_diagnostics()
    print("output_path:", output_path)
    print("last_epoch:", row)
    render_history(history)

print("saved_checkpoint:", output_path)


In [ ]:
from pathlib import Path
import shutil

drive_output_dir = Path(DRIVE_ARTIFACT_DIR)
drive_output_dir.mkdir(parents=True, exist_ok=True)
target_path = drive_output_dir / output_path.name
shutil.copy2(output_path, target_path)
print('copied to drive:', target_path)
